# Outliers Detection
### Techniques for Identifying and Handling Outliers in the Titanic Dataset

## 1. Project Overview
This notebook demonstrates multiple outlier detection techniques using the Titanic dataset. We explore statistical methods (IQR, Z-score), visual methods (box plots, scatter plots), and discuss when and how to handle outliers in real-world data.

## 2. Learning Objectives
- Understand what constitutes an outlier and why they matter
- Apply IQR-based outlier detection
- Use Z-score and modified Z-score methods
- Visualise outliers with box plots and distribution charts
- Compare the impact of outlier removal on statistical summaries

## 3. Business / Research Problem
**Question:** How can we systematically identify outliers in passenger data (age, fare, family size) and what is the appropriate treatment strategy for each feature?

## 4. Why This Analysis Matters
Outliers can distort statistical analyses, bias machine learning models, and lead to incorrect conclusions. Understanding detection methods and appropriate handling strategies is a fundamental data science skill.

## 5. Dataset Overview
Using the Titanic dataset with key columns:
- `passenger_id`, `pclass` — identifier and ticket class
- `age`, `fare` — continuous features likely to contain outliers
- `sibsp`, `parch` — family aboard counts
- `survived` — survival indicator
- `sex`, `embarked` — categorical features

## 6. Dataset Source and License Notes
- **Source:** Classic Titanic dataset (public domain)
- **File:** `titanic.csv`
- **License:** Public domain

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['pandas','numpy','matplotlib','seaborn','scipy']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
CSV_FILE = Path('titanic.csv')
IQR_MULTIPLIER = 1.5
ZSCORE_THRESHOLD = 3

## 10. Dataset Loading

In [ ]:
df = pd.read_csv(CSV_FILE)
print(f'Shape: {df.shape}')
df.head()

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print(f'\nMissing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f'\nNumeric columns:')
print(df.select_dtypes(include=[np.number]).columns.tolist())

## 12. Data Cleaning
1. Drop rows with missing age for outlier analysis
2. Focus on numeric columns relevant for outlier detection

In [ ]:
df_clean = df.dropna(subset=['age','fare']).copy()
print(f'Working dataset: {df_clean.shape}')
numeric_features = ['age', 'fare', 'sibsp', 'parch']
df_clean[numeric_features].describe()

## 13. Exploratory Data Analysis
### Initial Distribution Overview

In [ ]:
print('Fare statistics:')
print(df_clean['fare'].describe())
print(f'\nFare skewness: {df_clean["fare"].skew():.2f}')
print(f'Age skewness: {df_clean["age"].skew():.2f}')
print(f'\n99th percentile fare: ${df_clean["fare"].quantile(0.99):.2f}')

## 14. Univariate Analysis
### Box Plots — Visual Outlier Detection

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, col in zip(axes, numeric_features):
    sns.boxplot(data=df_clean, y=col, ax=ax, color='steelblue')
    ax.set_title(f'{col} — Box Plot')
plt.suptitle('Box Plots for Numeric Features', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Distribution plots with KDE
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_clean['age'].hist(bins=30, ax=axes[0], color='coral', edgecolor='white')
axes[0].set_title('Age Distribution')
df_clean['fare'].hist(bins=50, ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title('Fare Distribution')
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis
### How outliers relate to other features

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=df_clean, x='age', y='fare', hue='pclass', alpha=0.6, ax=axes[0], palette='viridis')
axes[0].set_title('Age vs Fare (colored by Class)')

sns.boxplot(data=df_clean, x='pclass', y='fare', ax=axes[1], palette='Set2')
axes[1].set_title('Fare by Passenger Class')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### IQR Method for Outlier Detection

In [ ]:
def detect_outliers_iqr(series, multiplier=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - multiplier * IQR
    upper = Q3 + multiplier * IQR
    outliers = series[(series < lower) | (series > upper)]
    return outliers, lower, upper

print('IQR-based outlier detection:')
print('=' * 50)
for col in numeric_features:
    outliers, lo, hi = detect_outliers_iqr(df_clean[col])
    print(f'\n{col}:')
    print(f'  Bounds: [{lo:.2f}, {hi:.2f}]')
    print(f'  Outliers: {len(outliers)} ({len(outliers)/len(df_clean)*100:.1f}%)')

## 17. Statistical Checks / Hypothesis Testing
### Z-Score Method

In [ ]:
print('Z-score outlier detection (|z| > 3):')
print('=' * 50)
for col in numeric_features:
    z = np.abs(stats.zscore(df_clean[col]))
    n_outliers = (z > ZSCORE_THRESHOLD).sum()
    print(f'{col}: {n_outliers} outliers ({n_outliers/len(df_clean)*100:.1f}%)')

# Compare IQR vs Z-score for fare
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fare = df_clean['fare']

# IQR
outliers_iqr, lo, hi = detect_outliers_iqr(fare)
axes[0].hist(fare, bins=50, color='lightblue', edgecolor='white')
axes[0].axvline(lo, color='red', linestyle='--', label=f'Lower: {lo:.0f}')
axes[0].axvline(hi, color='red', linestyle='--', label=f'Upper: {hi:.0f}')
axes[0].set_title(f'IQR Method — {len(outliers_iqr)} outliers')
axes[0].legend()

# Z-score
z = np.abs(stats.zscore(fare))
axes[1].hist(fare, bins=50, color='lightgreen', edgecolor='white')
threshold_val = fare[z <= ZSCORE_THRESHOLD].max()
axes[1].axvline(threshold_val, color='red', linestyle='--', label=f'Z=3 threshold')
axes[1].set_title(f'Z-score Method — {(z > ZSCORE_THRESHOLD).sum()} outliers')
axes[1].legend()

plt.tight_layout(); plt.show()

## 18. Visual Analysis
### Impact of Outlier Removal on Distributions

In [ ]:
# Remove fare outliers using IQR and compare distributions
outliers_iqr, lo, hi = detect_outliers_iqr(df_clean['fare'])
df_no_outliers = df_clean[(df_clean['fare'] >= lo) & (df_clean['fare'] <= hi)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_clean['fare'].hist(bins=40, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Fare — With Outliers (n={len(df_clean)})')
axes[0].set_xlabel('Fare')

df_no_outliers['fare'].hist(bins=40, ax=axes[1], color='seagreen', edgecolor='white')
axes[1].set_title(f'Fare — Outliers Removed (n={len(df_no_outliers)})')
axes[1].set_xlabel('Fare')

plt.tight_layout(); plt.show()

print(f'Mean fare — with outliers: ${df_clean["fare"].mean():.2f}')
print(f'Mean fare — without:       ${df_no_outliers["fare"].mean():.2f}')
print(f'Std fare — with outliers:   ${df_clean["fare"].std():.2f}')
print(f'Std fare — without:          ${df_no_outliers["fare"].std():.2f}')

## 19. Key Findings
1. **Fare has the most outliers** — a long right tail with luxury cabin fares exceeding $500.
2. **IQR detects more outliers** than Z-score for skewed distributions like fare.
3. **Age outliers are minimal** — the distribution is roughly normal.
4. **SibSp and Parch** have structural outliers (large families are rare but genuine).
5. **Outlier removal significantly reduces** the mean and standard deviation of fare.

## 20. Limitations
- Outlier ≠ error: Titanic first-class passengers genuinely paid high fares.
- No single method is universally best — domain context matters.
- Removing outliers can bias results by eliminating legitimate observations.
- Small dataset amplifies the effect of each removed point.

## 21. Recommendations / Next Steps
1. Apply domain-based rules (e.g., separate analysis by pclass) before flagging outliers.
2. Try robust methods: Isolation Forest, LOF (Local Outlier Factor).
3. Use winsorisation (capping) instead of removal when outliers are legitimate.
4. Evaluate how outlier treatment affects downstream model performance.
5. Document every outlier decision with justification.

## 22. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Removing all statistical outliers blindly | Some outliers are valid data | Use domain knowledge |
| Using Z-score on skewed data | Assumes normality | Use IQR or MAD instead |
| Not documenting outlier treatment | Irreproducible analysis | Log every decision |
| Applying outlier detection to categorical data | Meaningless for categories | Only use on continuous features |
| Checking outliers after scaling | Scaling changes distribution | Detect outliers on raw data first |

## 23. Mini Challenge / Exercises
1. **Modified Z-score:** Implement MAD-based outlier detection and compare results with IQR.
2. **Per-class analysis:** Detect fare outliers within each pclass separately — do results change?
3. **Visualisation:** Create a violin plot comparing fare distributions with and without outliers.
4. **Winsorisation:** Cap fare at the 95th percentile and compare the resulting distribution.
5. **Isolation Forest:** Apply sklearn's IsolationForest to all numeric features — which passengers are flagged?

## 24. Final Summary / Key Takeaways
```
TAKEAWAY 1  Outlier detection method should match data distribution (IQR for skewed, Z-score for normal).
TAKEAWAY 2  Fare is the most outlier-prone feature in the Titanic dataset.
TAKEAWAY 3  Context matters — first-class fares are outliers statistically but valid data points.
TAKEAWAY 4  IQR is more conservative than Z-score for skewed distributions.
TAKEAWAY 5  Always compare analysis results with and without outliers to assess their impact.
```